In [1]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re
import subprocess

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
R_PATH = local_config['R_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import data_tools as dt
import writing_tools as wt
from utils import parse_casenum
import utils

from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegression
from IPython.core.display import HTML
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import statsmodels.api as sm
from stargazer.stargazer import Stargazer

from scipy.spatial.distance import mahalanobis


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10



In [2]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_embeddings.parquet"))

In [3]:
# Council district dummies

vals = ['1','2','3','4','5','6','7','8','9','10','11','12','13','14','15','CITYWIDE']
for v in vals:
    df[f'cd_{v}'] = 0

invalids = []
for idx, row in df.iterrows():
    splitted = re.split(r"[,;]", row['council_district'])
    council_districts = [x.strip() for x in splitted if len(x.strip())>0]
    for cd in council_districts:
        if cd in vals:
            df.loc[idx, f'cd_{cd}'] = 1
        else:
            invalids.append((idx, cd))

print(invalids)

# apply fixes
df.loc[82, "cd_9"] = 1
df.loc[572, "cd_CITYWIDE"] = 1

[(82, 'Multiple'), (571, 'ALL')]


In [4]:
# Suffix encoding

sfx_df = pd.read_csv(os.path.join(LOCAL_PATH, "intermediate_data/cpc", "suffix_groups.csv"))
sfx_df = sfx_df.set_index("suffix")
groups = sfx_df['group'].unique().tolist()
for grp in groups:
    df[f"sfx_{grp}"] = 0

for idx, row in df.iterrows():
    title = row["title"]
    parsed_casenum = parse_casenum(title)
    suffixes = parsed_casenum['suffixes']
    for sfx in suffixes:
        grp = sfx_df.loc[sfx, 'group']
        df.at[idx, f"sfx_{grp}"] = 1


In [5]:
# Agenda order and number of agenda items

df["agenda_order"] = np.nan
for idx, row in df.iterrows():
    item_no = row["item_no"]
    agenda_order = int(re.match(r'\d+', item_no).group())
    df.at[idx, "agenda_order"] = agenda_order

df["num_agenda_items"] = df.groupby("date")["agenda_order"].transform("max")

In [6]:
# support and opposition

df["log2_support"] = np.log2(df["n_support"] + 1)
df["log2_oppose"] = np.log2(df["n_oppose"] + 1)

In [7]:
# Calculate mahalanobis distance

cols = [f'd{i}' for i in range(N_COMPONENTS)]
centroid_cols = [f'cluster_d{i}' for i in range(N_COMPONENTS)]

cov_invs = {}
for cluster, group in df.groupby('cluster'):
    cov = group[cols].cov().values
    cov_invs[cluster] = np.linalg.inv(cov)

df['mahalanobis'] = np.nan
for idx, row in df.iterrows():
    cluster = row['cluster']
    x = row[cols].values
    centroid = row[centroid_cols].values
    df.at[idx, 'mahalanobis'] = mahalanobis(x, centroid, cov_invs[cluster])


In [8]:
# Outcome variable

df["project_result"].value_counts()

df["outcome"] = 1
df.loc[df["project_result"] == "APPROVED", "outcome"] = 2
df.loc[df["project_result"].isin(['DELAYED', 'DENIED', 'APPLICATION WIDTHDRAWN']), "outcome"] = 0


In [9]:
# Output regression data

df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_data.parquet"))

In [10]:
# Running R regressions

res = subprocess.run([R_PATH, LOCAL_PATH + "/src/R/15-ordered-logit.R"], check=True, capture_output=True, text=True)
print(res.stdout)


                             Dependent variable:          
                    --------------------------------------
                                   outcome                
                       (1)       (2)       (3)      (4)   
----------------------------------------------------------
mahalanobis         -0.347*** -0.313*** -0.323*** -0.283**
                     (0.095)   (0.097)   (0.102)  (0.112) 
                                                          
agenda_order                   0.090*    0.088*    0.059  
                               (0.048)   (0.049)  (0.051) 
                                                          
num_agenda_items               -0.026    -0.017    0.001  
                               (0.040)   (0.041)  (0.043) 
                                                          
is_consent_calendar           1.442***  1.460***  1.477***
                               (0.253)   (0.265)  (0.273) 
                                                       

In [11]:
# Load regression coefficients

coefs_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_coefs.parquet"))


In [12]:
# Output table

header = r"""\begin{table}[H]
\centering
\caption{Ordered Logit Regression Results}
\vspace{0.2cm}
\label{tab_ologit_results}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lcccc}
\toprule
 & (1) & (2) & (3) & (4) \\
\midrule
 &  &  &  &  \\
"""
footer = r"""\bottomrule
\end{tabular}
\begin{tablenotes}[flushleft]
\footnotesize
\item Robust standard errors in parentheses. * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
\item \textit{Notes:} This table reports coefficient estimates from the ordered logit regression described in Section \ref{sec_methodology}.
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
reg_names = ["r1", "r2", "r3","r4"]

vars = [
    ("mahalanobis", "Semantic Uniqueness"),
    ("agenda_order", "Agenda Order"),
    ("num_agenda_items", "No. Agenda Items"),
    ("is_consent_calendarTRUE", "Consent Calendar"),
    ("log2_support", "$\\log_2$(\\# Support)"),
    ("log2_oppose", "$\\log_2$(\\# Oppose)")
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += r"""& & & & \\
Semantic Cluster FE         & N & Y & Y & Y \\
Council District FE         & N & N & Y & Y \\
Case Suffix Group FE        & N & N & N & Y \\
& & & & \\
"""

vars = [
    ("0|1", "$\\mu_0$"),
    ("1|2", "$\\mu_1$")
]

for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ [1.8ex]" + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "tables", "tab_ologit_results.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)


\begin{table}[H]
\centering
\caption{Ordered Logit Regression Results}
\vspace{0.2cm}
\label{tab_ologit_results}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lcccc}
\toprule
 & (1) & (2) & (3) & (4) \\
\midrule
 &  &  &  &  \\
Semantic Uniqueness  & -0.347$^{***}$ & -0.313$^{***}$ & -0.323$^{***}$ & -0.283$^{**}$ \\
 & (0.095) & (0.097) & (0.102) & (0.112) \\ [1.8ex]
Agenda Order  &  & 0.090$^{*}$ & 0.088$^{*}$ & 0.059$^{}$ \\
 &  & (0.048) & (0.049) & (0.051) \\ [1.8ex]
No. Agenda Items  &  & -0.026$^{}$ & -0.017$^{}$ & 0.001$^{}$ \\
 &  & (0.040) & (0.041) & (0.043) \\ [1.8ex]
Consent Calendar  &  & 1.442$^{***}$ & 1.460$^{***}$ & 1.477$^{***}$ \\
 &  & (0.253) & (0.265) & (0.273) \\ [1.8ex]
$\log_2$(\# Support)  &  & 0.054$^{}$ & 0.055$^{}$ & 0.056$^{}$ \\
 &  & (0.060) & (0.062) & (0.063) \\ [1.8ex]
$\log_2$(\# Oppose)  &  & -0.130$^{**}$ & -0.149$^{***}$ & -0.137$^{**}$ \\
 &  & (0.055) & (0.057) & (0.058) \\ [1.8ex]
& & & & \\
Semantic Cluster FE

In [13]:
# Load marginals

marginals_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_marginals.parquet"))

In [20]:
# Marginal effects table

header = r"""\begin{table}[H]
\centering
\caption{Ordered Logit Marginal Effects}
\vspace{0.2cm}
\label{tab_ologit_marginal_effects}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lccc}
\toprule
 & (1) & (2) & (3) \\
\midrule
 & Outcome 0 & Outcome 1 & Outcome 2 \\
 &  &  &  \\
"""
footer = r"""\bottomrule
\end{tabular}
\begin{tablenotes}[flushleft]
\footnotesize
\item Robust standard errors in parentheses. * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
\item \textit{Notes:} This table reports estimated marginal effects from the ordered logit regression in column (4) of Table \ref{tab_ologit_results}.
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
outcomes = ["0", "1", "2"]

vars = [
    ("mahalanobis", "Semantic Uniqueness"),
    ("agenda_order", "Agenda Order"),
    ("num_agenda_items", "No. Agenda Items"),
    ("is_consent_calendar", "Consent Calendar"),
    ("log2_support", "$\\log_2$(\\# Support)"),
    ("log2_oppose", "$\\log_2$(\\# Oppose)")
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for y in outcomes:
        idx = (marginals_df["regression_name"]=="r4") & (marginals_df["term"]==v[0]) & (marginals_df["group"]==y)
        if idx.sum()==0:
            tbl += " & "
            continue
        meff = marginals_df.loc[idx, "estimate"].values[0]
        serr = marginals_df.loc[idx, "std.error"].values[0]
        stars = utils.stars(meff, serr)
        tbl += f" & {meff:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for y in outcomes:
        idx = (marginals_df["regression_name"]=="r4") & (marginals_df["term"]==v[0]) & (marginals_df["group"]==y)
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = marginals_df.loc[idx, "std.error"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += r"""& & & \\
Semantic Cluster FE         & Y & Y & Y \\
Council District FE         & Y & Y & Y \\
Case Suffix Group FE        & Y & Y & Y \\
& & & \\
"""

tbl += "Observations "
for y in outcomes:
    nobs = (df["outcome"]==int(y)).sum()
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ [1.8ex]" + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "tables", "tab_ologit_marginal_effects.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)


\begin{table}[H]
\centering
\caption{Ordered Logit Marginal Effects}
\vspace{0.2cm}
\label{tab_ologit_marginal_effects}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lccc}
\toprule
 & (1) & (2) & (3) \\
\midrule
 & Outcome 0 & Outcome 1 & Outcome 2 \\
 &  &  &  \\
Semantic Uniqueness  & 0.035$^{**}$ & 0.025$^{**}$ & -0.060$^{**}$ \\
 & (0.014) & (0.010) & (0.023) \\ [1.8ex]
Agenda Order  & -0.007$^{}$ & -0.005$^{}$ & 0.012$^{}$ \\
 & (0.006) & (0.004) & (0.011) \\ [1.8ex]
No. Agenda Items  & -0.000$^{}$ & -0.000$^{}$ & 0.000$^{}$ \\
 & (0.005) & (0.004) & (0.009) \\ [1.8ex]
Consent Calendar  & -0.132$^{***}$ & -0.190$^{***}$ & 0.322$^{***}$ \\
 & (0.019) & (0.039) & (0.053) \\ [1.8ex]
$\log_2$(\# Support)  & -0.007$^{}$ & -0.005$^{}$ & 0.012$^{}$ \\
 & (0.008) & (0.006) & (0.013) \\ [1.8ex]
$\log_2$(\# Oppose)  & 0.017$^{**}$ & 0.012$^{**}$ & -0.029$^{**}$ \\
 & (0.007) & (0.005) & (0.012) \\ [1.8ex]
& & & \\
Semantic Cluster FE         & Y & Y & Y \\
C